## Raw data sets treatment to create useable new spectral data bases

In [ ]:
import csv
from pathlib import Path
import pandas as pd
from IPython.display import display

# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
simplefilter(action='ignore', category=UserWarning)
simplefilter(action="ignore", category=RuntimeWarning)

In [2]:
# Function to load a CSV file with automatic separator detection

def load_csv_auto_sep(mode, data_source, type_data, verbose=True, delimiter=None):

    ## Importation of the datasets with the adapted path
    file_name = Path("Data/%s/%s"% (mode,data_source))
    full_path = str(file_name.resolve()).replace("\\", "/")
    path = full_path + "/%s.csv" % type_data
    
    with open(path, 'r', newline='', encoding='utf-8-sig') as f:

        if delimiter is not None:
            sep = delimiter
        
        else:
            # Read a small portion of the file to detect the separator
            excerpt = f.read(1024)
            f.seek(0)  # return to the beginning of the file

            # Detection of the dialect
            dialect = csv.Sniffer().sniff(excerpt)
            sep = dialect.delimiter

        if verbose: print("Detected separator for %s: %s" % (type_data, sep))
        
        # Load the file with pandas
        df = pd.read_csv(f, delimiter=sep)

        if type_data[0]=='Y' and len(df.columns) > 1:
            # Drop the useless column if it exists
            df = df.drop(columns=[df.columns[1]])
        
        return df

## Grapevines

In [ ]:
df_nirs = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="innospectra_reflectance", verbose=True, delimiter=None)
df_nirs = df_nirs.drop(columns=["Unnamed: 0"])
df1 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings", verbose=True, delimiter=None)
df2 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings (1)", verbose=True, delimiter=None)
df = pd.concat([df1, df2], axis=0)
df = df.rename(columns={"svc_id": "scan"})
display(df.head())

Detected separator for innospectra_reflectance: ,
Detected separator for chloridometer_readings: ,
Detected separator for chloridometer_readings (1): ,


,pot number,scan,genotype,rep,trt,reading1,reading2,reading3,average
0,1,HR.060623.0090.sig,LONGII9035,5,75,33.0,32.0,33.0,921.485661
1,2,HR.060623.0089.sig,BERL9031,5,75,115.0,118.0,117.0,3291.020217
2,3,HR.060623.0088.sig,NM11-081,2,75,148.0,148.0,150.0,4193.700048
3,4,HR.060623.0087.sig,NM11-085,3,75,198.0,199.0,198.0,5594.734367
4,5,HR.060623.0086.sig,AZ11-012B,2,75,121.0,121.0,120.0,3403.855196


In [ ]:
import os
import pandas as pd
import re

# Chemin du dossier contenant les fichiers CSV
path = 'Data/Raw/Grapevines_chloride'  # ← À modifier selon ton cas

# Expression régulière pour capturer les fichiers au bon format
pattern = re.compile(r'(\d{6})_svc_reflectance\.csv')

# Dictionnaire pour stocker les DataFrames, clés = dates
dataframes = {}

# Parcours des fichiers du dossier
for fichier in os.listdir(path):
    match = pattern.match(fichier)
    if match:
        date_str = match.group(1)  # Extrait la date 'yymmdd'
        chemin_fichier = os.path.join(path, fichier)
        df = pd.read_csv(chemin_fichier)
        dataframes[date_str] = df

# Exemple : afficher les dates chargées
print("Fichiers chargés :", list(dataframes.keys()))
for date_str, df in dataframes.items():
    print(f"Date : {date_str}, Nombre de lignes : {len(df)}")

all_data = []

for fichier in os.listdir(path):
    match = pattern.match(fichier)
    if match:
        date_str = match.group(1)
        chemin_fichier = os.path.join(path, fichier)
        df = pd.read_csv(chemin_fichier)
        df['date'] = pd.to_datetime(date_str, format='%y%m%d')
        all_data.append(df)

df_total = pd.concat(all_data, ignore_index=True)
display(df_total)


,scan,338.9,340.4,341.9,343.3,344.8,346.3,347.7,349.2,350.7,...,2498.7,2500.8,2502.9,2505,2507,2509.1,2511.2,2513.3,2515.3,date
0,HR.062623.0000.sig,9.84,6.67,9.58,8.72,9.64,9.33,6.88,5.31,6.30,...,6.26,6.01,5.71,5.58,5.50,5.72,5.97,5.88,6.40,2023-06-26
1,HR.062623.0001.sig,1.99,5.51,12.80,10.88,5.11,4.71,3.91,8.04,7.30,...,5.99,5.75,5.62,5.87,5.98,5.72,5.56,6.09,6.51,2023-06-26
2,HR.062623.0002.sig,14.43,8.48,6.20,3.59,7.71,10.22,7.74,4.90,3.33,...,4.93,4.85,4.51,4.64,5.02,4.93,4.25,4.44,5.25,2023-06-26
3,HR.062623.0003.sig,6.95,8.26,6.83,6.99,7.30,5.39,6.52,2.47,7.30,...,5.73,5.66,5.80,5.77,5.60,5.52,5.56,5.68,5.25,2023-06-26
4,HR.062623.0004.sig,4.97,9.18,5.98,6.22,7.30,4.04,3.91,5.57,3.93,...,6.26,6.01,5.89,6.15,5.79,5.62,5.86,5.68,6.51,2023-06-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1500,HR.050923.0298.sig,8.33,15.28,15.25,5.86,7.61,4.93,3.75,5.17,6.45,...,3.44,3.25,3.26,3.26,3.08,3.06,3.23,3.48,3.45,2023-05-09
1501,HR.050923.0299.sig,8.89,11.67,8.14,7.50,7.39,9.08,8.63,7.93,5.16,...,3.52,3.33,3.43,3.61,3.43,3.51,3.59,3.38,3.55,2023-05-09
1502,HR.050923.0300.sig,11.11,9.72,8.90,8.20,8.70,7.89,7.50,7.76,9.68,...,4.48,4.47,4.76,4.64,4.31,4.68,4.88,4.61,5.18,2023-05-09
1503,HR.050923.0301.sig,12.50,15.28,12.71,8.20,7.61,5.92,8.44,6.90,7.26,...,4.88,5.04,5.10,4.72,4.75,4.86,4.88,4.42,3.93,2023-05-09


In [35]:
ids_communs = set(df_total['scan']).intersection(df['scan'])
df_total_filtré = df_total[df_total['scan'].isin(ids_communs)].reset_index(drop=True)
df_filtré = df[df['scan'].isin(ids_communs)].reset_index(drop=True)
display(df_total_filtré)
display(df_filtré)
Y = df_filtré['average']
Y.min(), Y.max(), Y.mean()

,scan,338.9,340.4,341.9,343.3,344.8,346.3,347.7,349.2,350.7,...,2498.7,2500.8,2502.9,2505,2507,2509.1,2511.2,2513.3,2515.3,date
0,HR.071823.0000.sig,8.33,4.76,6.15,9.86,13.16,10.34,4.40,2.11,4.95,...,5.31,4.97,4.98,4.74,4.69,4.79,4.91,4.91,4.88,2023-07-18
1,HR.071823.0001.sig,8.00,6.98,8.62,9.58,5.26,6.44,5.71,6.32,4.36,...,3.94,4.36,4.27,4.20,4.03,3.93,4.02,4.11,3.76,2023-07-18
2,HR.071823.0002.sig,16.67,12.70,12.31,2.82,2.63,8.05,8.79,12.63,13.86,...,6.76,6.71,6.49,6.39,6.56,6.81,6.67,6.41,6.71,2023-07-18
3,HR.071823.0003.sig,13.13,14.43,6.53,5.12,6.38,6.27,6.66,10.85,8.40,...,5.91,5.75,5.42,5.84,5.90,5.75,5.79,5.61,5.60,2023-07-18
4,HR.071823.0004.sig,12.12,12.51,11.19,8.54,9.57,10.45,6.66,7.66,7.80,...,4.11,4.44,4.53,4.20,3.94,3.84,3.93,4.21,3.87,2023-07-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
551,HR.060623.0305.sig,7.65,14.15,11.64,8.90,8.08,6.52,8.88,10.31,7.06,...,5.35,5.30,5.33,5.08,4.93,5.17,5.19,5.00,5.62,2023-06-06
552,HR.060623.0306.sig,11.13,11.15,6.27,9.24,7.69,8.56,8.97,10.80,10.70,...,5.94,5.64,5.42,5.62,5.48,5.36,5.68,5.70,5.31,2023-06-06
553,HR.060623.0307.sig,12.06,12.01,7.84,8.47,7.69,9.22,6.58,6.25,4.28,...,4.16,4.17,4.27,4.17,4.00,4.21,4.31,3.90,4.09,2023-06-06
554,HR.060623.0308.sig,8.35,6.86,4.70,9.24,7.69,8.56,10.17,6.25,6.95,...,5.94,5.73,5.78,5.98,5.86,5.65,5.78,6.00,6.03,2023-06-06


,pot number,scan,genotype,rep,trt,reading1,reading2,reading3,average
0,1,HR.060623.0090.sig,LONGII9035,5,75,33.0,32.0,33.0,921.485661
1,2,HR.060623.0089.sig,BERL9031,5,75,115.0,118.0,117.0,3291.020217
2,3,HR.060623.0088.sig,NM11-081,2,75,148.0,148.0,150.0,4193.700048
3,4,HR.060623.0087.sig,NM11-085,3,75,198.0,199.0,198.0,5594.734367
4,5,HR.060623.0086.sig,AZ11-012B,2,75,121.0,121.0,120.0,3403.855196
...,...,...,...,...,...,...,...,...,...
551,320,HR.071823.0199.sig,LONGII9018,2,0,5.0,6.0,6.0,159.849553
552,321,HR.071823.0200.sig,RAMSEY,1,0,7.0,8.0,8.0,216.267043
553,322,HR.071823.0201.sig,RAMSEY,2,0,5.0,6.0,6.0,159.849553
554,323,HR.071823.0202.sig,BERL9031,5,0,11.0,11.0,10.0,300.893277


(np.float64(0.0), np.float64(7945.463094), np.float64(1664.129909058795))

In [37]:
ids_communs = set(df_nirs['pot number']).intersection(df1['pot number'])
df_nirs_filtered = df_nirs[df_nirs['pot number'].isin(ids_communs)].reset_index(drop=True)
df1_filtered = df1[df1['pot number'].isin(ids_communs)].reset_index(drop=True)
display(df_nirs_filtered)
display(df1_filtered)
Y = df1_filtered['average']
Y.min(), Y.max(), Y.mean()

,Unnamed: 0,pot number,X901,X905,X909,X913,X917,X922,X926,X930,...,X1673,X1677,X1680,X1683,X1686,X1689,X1692,X1695,X1698,X1702
0,1,1,0.585469,0.588581,0.585256,0.586831,0.584548,0.575164,0.574076,0.570778,...,0.493503,0.513089,0.536680,0.548472,0.572834,0.583946,0.585639,0.588452,0.579730,0.564959
1,10,10,0.566303,0.572478,0.569457,0.567158,0.564576,0.562383,0.557970,0.558688,...,0.476566,0.490628,0.505247,0.510241,0.528924,0.534094,0.530099,0.536508,0.537251,0.525435
2,102,102,0.554481,0.551405,0.557599,0.555695,0.556986,0.552940,0.551291,0.552251,...,0.464455,0.471107,0.469504,0.479313,0.477919,0.483467,0.482062,0.487115,0.483292,0.474390
3,103,103,0.529521,0.525454,0.529093,0.528698,0.530865,0.526186,0.529362,0.528061,...,0.413945,0.416315,0.415728,0.420411,0.422865,0.423340,0.430078,0.424106,0.425015,0.422554
4,104,104,0.564320,0.566698,0.569917,0.568110,0.570216,0.569071,0.568365,0.567343,...,0.410190,0.412934,0.412438,0.416779,0.414624,0.422081,0.424782,0.421083,0.414841,0.412059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,95,95,0.526489,0.528208,0.529416,0.528177,0.528488,0.528560,0.532504,0.530270,...,0.436586,0.438722,0.439840,0.444017,0.445941,0.455646,0.448858,0.445552,0.452110,0.454927
256,96,96,0.622445,0.623665,0.624592,0.621484,0.618666,0.620824,0.621720,0.619274,...,0.487451,0.488841,0.489330,0.488344,0.485779,0.488932,0.487590,0.483970,0.486359,0.483723
257,97,97,0.577716,0.585404,0.581876,0.585473,0.582522,0.581177,0.581627,0.579099,...,0.438764,0.440492,0.440387,0.444217,0.444126,0.445287,0.439567,0.450635,0.447502,0.433557
258,98,98,0.616947,0.623122,0.619884,0.620664,0.618463,0.617026,0.616805,0.616563,...,0.431388,0.433526,0.431759,0.430005,0.425132,0.431901,0.429218,0.429081,0.436841,0.424370


,pot number,svc_id,genotype,rep,trt,reading1,reading2,reading3,average
0,1,HR.060623.0090.sig,LONGII9035,5,75,33.0,32.0,33.0,921.485661
1,2,HR.060623.0089.sig,BERL9031,5,75,115.0,118.0,117.0,3291.020217
2,3,HR.060623.0088.sig,NM11-081,2,75,148.0,148.0,150.0,4193.700048
3,4,HR.060623.0087.sig,NM11-085,3,75,198.0,199.0,198.0,5594.734367
4,5,HR.060623.0086.sig,AZ11-012B,2,75,121.0,121.0,120.0,3403.855196
...,...,...,...,...,...,...,...,...,...
255,262,HR.060623.0269.sig,140RU,5,0,6.0,5.0,4.0,141.043724
256,263,HR.060623.0268.sig,GIRDIANA6B,5,0,8.0,7.0,8.0,216.267043
257,264,HR.060623.0267.sig,AZ11-012B,1,0,5.0,5.0,3.0,122.237894
258,265,HR.060623.0266.sig,NM11-081,3,0,5.0,5.0,3.0,122.237894


(np.float64(0.0), np.float64(7936.060178), np.float64(1570.8654298436536))

## Milk

In [46]:
df = load_csv_auto_sep(mode="Raw", data_source="milk", type_data="data_table", verbose=True, delimiter=None)
display(df.head())
df['Urea'].mean()

Detected separator for data_table: ,


,Cow_ID,Fat,Prot,Lact,SCC,Urea,Milk_yield,Milk_Interv,SET,Time_Dark,...,Trans_White_247,Trans_White_248,Trans_White_249,Trans_White_250,Trans_White_251,Trans_White_252,Trans_White_253,Trans_White_254,Trans_White_255,Trans_White_256
0,57017,2.79,3.55,4.84,87,30,12.73,27256,1,1495647915,...,22039,21978,21916,21872,21818,21780,21758,21720,21698,21672
1,53330,4.70,3.29,4.94,14,29,15.00,34359,1,1495648408,...,22039,21978,21915,21872,21817,21779,21757,21719,21697,21672
2,59129,3.35,3.21,4.96,21,25,13.26,33081,1,1495648930,...,22039,21978,21916,21872,21817,21780,21758,21720,21698,21672
3,53333,2.69,3.02,4.83,9,31,18.19,30596,1,1495649484,...,22040,21979,21916,21872,21818,21780,21758,21720,21698,21672
4,57013,4.60,3.83,4.70,14,28,13.22,39668,1,1495651684,...,22039,21978,21915,21871,21816,21779,21757,21719,21696,21671


np.float64(27.598856209150327)

## Manure

In [ ]:
# Read the csv file
df = load_csv_auto_sep(mode="Raw", data_source="manure", type_data="spectra_standard_cells", verbose=False, delimiter=None)

# Read the xlsx file
df_chem = pd.read_excel("Data/Raw/manure/chemical_analysis.xlsx")
display(df_chem)

# Read the xlsx file
df_nirs_dry = pd.read_excel("Data/Raw/manure/spectra_DG_Abs_STD_1100_2498nm_STD.xlsx")
display(df_nirs_dry)


df_nirs_fresh = pd.read_excel("Data/Raw/manure/spectra_FH_Abs_STD_1100_2498nm_STD.xlsx")
display(df_nirs_fresh)

,sample_name,DM,NH4,N,CaO,K2O,MgO,P2O5,type_manure,spectrometer,township,country
0,FBCRIF001,20.815,0.075,0.490,NaN,NaN,NaN,NaN,cattle manure,NIRFlex LDAR,DERVAL,mainland France
1,FBCRIF002,25.480,0.182,0.607,0.157,0.791,0.079,0.131,cattle manure,NIRFlex LDAR,DERVAL,mainland France
2,FBCRIF003,17.595,0.203,0.546,0.208,0.690,0.094,0.160,cattle manure,NIRFlex LDAR,DERVAL,mainland France
3,FBCRIF004,19.110,0.091,0.482,0.260,0.542,0.121,0.187,cattle manure,NIRFlex LDAR,CHATEAUNEUF DU FAOU,mainland France
4,FBCRIF005,18.600,0.089,0.458,NaN,NaN,NaN,NaN,cattle manure,NIRFlex LDAR,CHATEAUNEUF DU FAOU,mainland France
...,...,...,...,...,...,...,...,...,...,...,...,...
327,R_FERM_48_FBV_LT,37.350,0.052,0.632,NaN,NaN,NaN,NaN,cattle manure,XDS FOSS CIRAD,TROIS BASSINS,Reunion Island
328,R_FERM_49_FBV_LT,34.340,0.045,0.432,NaN,NaN,NaN,NaN,cattle manure,XDS FOSS CIRAD,TROIS BASSINS,Reunion Island
329,R_FERM_50_FBV_LT,25.690,0.068,0.536,NaN,NaN,NaN,NaN,cattle manure,XDS FOSS CIRAD,SAINT BENOIT,Reunion Island
330,R_FERM_69_FBV,30.660,0.154,0.642,NaN,NaN,NaN,NaN,cattle manure,XDS FOSS CIRAD,BOURG MURAT,Reunion Island


,sample_name,1100_nm,1102_nm,1104_nm,1106_nm,1108_nm,1110_nm,1112_nm,1114_nm,1116_nm,...,2480_nm,2482_nm,2484_nm,2486_nm,2488_nm,2490_nm,2492_nm,2494_nm,2496_nm,2498_nm
0,FBCRIF001,0.117115,0.116163,0.115307,0.114564,0.113552,0.112556,0.111460,0.110673,0.110249,...,0.352863,0.354768,0.356484,0.357876,0.359046,0.359973,0.360660,0.361052,0.361174,0.361169
1,FBCRIF002,0.127267,0.126541,0.125739,0.125051,0.124148,0.123225,0.122289,0.121577,0.121105,...,0.362040,0.364098,0.365936,0.367408,0.368604,0.369494,0.370083,0.370328,0.370269,0.370059
2,FBCRIF003,0.146101,0.145503,0.144815,0.144060,0.143048,0.142062,0.140952,0.140115,0.139674,...,0.380576,0.382347,0.383936,0.385200,0.386232,0.387005,0.387527,0.387755,0.387724,0.387581
3,FBCRIF004,0.145878,0.145194,0.144455,0.143708,0.142787,0.141831,0.140752,0.139927,0.139478,...,0.370933,0.372936,0.374728,0.376168,0.377350,0.378252,0.378879,0.379182,0.379185,0.379029
4,FBCRIF005,0.122828,0.122174,0.121568,0.121049,0.120233,0.119437,0.118623,0.117968,0.117673,...,0.361757,0.363635,0.365298,0.366606,0.367655,0.368422,0.368911,0.369075,0.368951,0.368688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327,R_FERM_48_FBV_LT,0.537676,0.537457,0.537198,0.536943,0.536700,0.536431,0.536126,0.535814,0.535520,...,0.624336,0.625337,0.626270,0.627058,0.627718,0.628293,0.628763,0.629069,0.629245,0.629400
328,R_FERM_49_FBV_LT,0.692955,0.692954,0.692958,0.692965,0.692965,0.692937,0.692876,0.692799,0.692728,...,0.747222,0.747918,0.748600,0.749176,0.749680,0.750098,0.750421,0.750639,0.750764,0.750878
329,R_FERM_50_FBV_LT,0.688883,0.688761,0.688609,0.688454,0.688296,0.688109,0.687887,0.687651,0.687433,...,0.754622,0.755523,0.756424,0.757197,0.757855,0.758463,0.758965,0.759284,0.759486,0.759662
330,R_FERM_69_FBV,0.262155,0.261556,0.260825,0.260102,0.259437,0.258782,0.258105,0.257412,0.256728,...,0.555208,0.557367,0.559325,0.561040,0.562519,0.563740,0.564675,0.565271,0.565539,0.565593


,sample_name,1100_nm,1102_nm,1104_nm,1106_nm,1108_nm,1110_nm,1112_nm,1114_nm,1116_nm,...,2480_nm,2482_nm,2484_nm,2486_nm,2488_nm,2490_nm,2492_nm,2494_nm,2496_nm,2498_nm
0,FBCRIF001,0.588418,0.588407,0.588012,0.587727,0.587602,0.587279,0.586653,0.586186,0.586046,...,1.684603,1.686953,1.689275,1.691759,1.694338,1.696744,1.698854,1.700564,1.701765,1.702570
1,FBCRIF002,0.530671,0.530607,0.530356,0.530160,0.530084,0.529749,0.529146,0.528818,0.528807,...,1.576865,1.579084,1.581248,1.583488,1.585777,1.587941,1.589935,1.591674,1.593029,1.594062
2,FBCRIF003,0.532509,0.532652,0.532556,0.532679,0.532500,0.532282,0.531813,0.531593,0.531658,...,1.610761,1.612479,1.614203,1.616133,1.618259,1.620364,1.622323,1.623984,1.625191,1.626028
3,FBCRIF004,0.495200,0.494887,0.494335,0.493886,0.493594,0.493283,0.492497,0.492086,0.492000,...,1.566656,1.568502,1.570339,1.572338,1.574459,1.576464,1.578253,1.579747,1.580875,1.581749
4,FBCRIF005,0.476230,0.476494,0.476465,0.476356,0.476468,0.476366,0.476144,0.476265,0.476501,...,1.601588,1.603243,1.604913,1.606800,1.608894,1.610958,1.612836,1.614357,1.615386,1.616054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327,R_FERM_48_FBV_LT,0.921903,0.921882,0.921879,0.921892,0.921893,0.921852,0.921772,0.921675,0.921602,...,1.521924,1.525306,1.528573,1.531451,1.534228,1.537057,1.539675,1.541941,1.544031,1.546002
328,R_FERM_49_FBV_LT,1.226104,1.226319,1.226631,1.226985,1.227300,1.227568,1.227816,1.228038,1.228266,...,1.557528,1.560118,1.562692,1.565166,1.567635,1.570050,1.572208,1.574032,1.575824,1.577748
329,R_FERM_50_FBV_LT,1.208262,1.208337,1.208493,1.208694,1.208869,1.208999,1.209085,1.209125,1.209165,...,1.782003,1.786263,1.790263,1.793536,1.796581,1.799726,1.802921,1.805966,1.809047,1.812130
330,R_FERM_69_FBV,0.571057,0.570358,0.569508,0.568678,0.567928,0.567190,0.566426,0.565650,0.564924,...,1.547658,1.550831,1.553917,1.556775,1.559369,1.561662,1.563666,1.565651,1.567573,1.569299


In [27]:
df_filtered = df_chem[(df_chem['type_manure'] == 'poultry manure') & (df_chem['spectrometer'] == "NIRFlex LDAR")]
print(len(df_filtered))
df_filtered.N.min(), df_filtered.N.max(), df_filtered.N.mean()

49


(np.float64(1.39), np.float64(3.739), np.float64(2.769979591836735))

## else

In [45]:
from scipy.io import loadmat
from scipy.io import loadmat
import numpy as np

# Chargement du fichier
data = loadmat('Data/Raw/mat/CGL_nir.mat')

# Récupérer l'objet 'Spectra'
spectra_struct = data['Spectra']

# Afficher les noms de champs disponibles
print("Champs disponibles dans 'Spectra' :", dir(spectra_struct))

# Hypothèse : les champs s'appellent souvent 'data', 'wavelength' ou similaire
# Affichons quelques détails
try:
    spectra_array = spectra_struct.data  # ou spectra_struct.y si nécessaire
    wavelengths = spectra_struct.x  # ou .wavelength, selon le nom exact

    print("Spectra shape:", spectra_array.shape)
    print("Wavelengths shape:", wavelengths.shape)

    # Stack sous forme (échantillons, longueurs d’onde)
    # selon orientation : (wavelengths,) x (spectra,) ou l'inverse
    if spectra_array.shape[0] == wavelengths.shape[0]:
        spectra_np = np.array(spectra_array)
    else:
        spectra_np = np.array(spectra_array).T  # transposé si nécessaire

    print("Final stacked array shape (spectra x wavelengths):", spectra_np.shape)

except AttributeError as e:
    print("Impossible d'accéder aux champs : ", e)


Champs disponibles dans 'Spectra' : ['T', '__abs__', '__add__', '__and__', '__array__', '__array_finalize__', '__array_function__', '__array_interface__', '__array_namespace__', '__array_priority__', '__array_struct__', '__array_ufunc__', '__array_wrap__', '__bool__', '__buffer__', '__class__', '__class_getitem__', '__complex__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__dlpack__', '__dlpack_device__', '__doc__', '__eq__', '__float__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__ilshift__', '__imatmul__', '__imod__', '__imul__', '__index__', '__init__', '__init_subclass__', '__int__', '__invert__', '__ior__', '__ipow__', '__irshift__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lshift__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__